# 02 — NBA Player Salaries Retrieval
### Basketball Reference | 2016-17 through 2025-26

This notebook scrapes player salary data from Basketball Reference across all 30 teams for 10 seasons (2016-17 through 2025-26), cleans and normalizes it against the salary cap, engineers contract-status features, and exports to a analysis-ready CSV.

**Source:** Basketball Reference Team Pages (`https://www.basketball-reference.com/teams`) — individual team season pages (e.g. `/teams/LAL/2024.html`)

**Output:** `nba_player_salaries.csv`

**Sections:** Setup → Scrape Single Team → Scrape All Seasons → Clean & Preprocess → Feature Engineering → Export

**Dependencies:** `requests`, `bs4`, `pandas`

### Output Schema

| Column | Type | Description |
|---|---|---|
| `player` | str | Full player name |
| `team` | str | BBRef 3-letter code; `TOT` if player switched teams mid-season |
| `year` | int | Season ending year (e.g. `2022` = 2021-22) |
| `season` | str | Human-readable label (e.g. `2021-22`) |
| `salary` | int | Player salary in USD |
| `salary_cap` | float | NBA salary cap for that season |
| `salary_cap_pct` | float | Salary as a percentage of that season's cap |
| `salary_rank` | int | Salary rank within the team that season |
| `salary_tier` | str | `bench` / `starter` / `star` / `max` based on cap % |
| `salary_pct_change` | float | Year-over-year salary change percentage |
| `is_max` | bool | True if salary ≥ 25% of the cap |
| `is_new_team` | bool | True if team differs from the prior season |
| `is_contract_year` | bool | True if a new deal was signed the following season |
| `is_moved_midseason` | bool | True if player was traded/moved mid-season (`team == TOT`) |

## Section 1 — Setup & Configuration

All imports, constants, and lookup tables used throughout the notebook. Defines the scrape range (`START_YEAR` / `END_YEAR`), the 30 Basketball Reference team abbreviations, salary cap values by season (sourced from Basketball Reference's cap history page), and the `SALARY_CAP_THRESHOLD_PCT` filter that removes minimum/two-way contract players in preprocessing.

In [1]:
%pip install --upgrade requests bs4 pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 55.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.5
    Uninstalling requests-2.32.5:
      Successfully uninstalled requests-2.32.5
  Attempting uninstall: pandas
    Found existing installation: pandas 2.1.4
    Uninstalling pandas-2.1.4:
      Successfully uninstalled pandas-2.1.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-sql-connector 4.2.4 requires pandas<2.4.0,>=1.2.5; python_version >= "3.8" and python_version < "3.13", but you have pandas 3.0.3 which is incompatible.
deepnote-toolkit 2.3.1 requires cryptography<47,>=46.0.5, but you have cryptography 44.0.3 which is incompati

In [1]:
from bs4 import BeautifulSoup, Comment
import requests
import pandas as pd
import time
import random
import unicodedata

In [27]:
# Base URL for Basketball Reference team pages
BASE_URL = 'https://www.basketball-reference.com/teams'

# Define headers to mimic a real browser request
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Define filepath for output file with cleaned NBA salary data
OUTPUT_FILE = './data/nba_player_salaries.csv'

# Scrape data for the last 10 seasons (2017-2026) to capture recent trends and contract year effects
START_YEAR, END_YEAR = 2017, 2026

# List of NBA team abbreviations
TEAMS = [
    'ATL', 'BOS', 'BRK', 'CHI', 'CHO', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 
    'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 
    'OKC', 'ORL', 'PHI', 'PHO', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS'
]

# Salary cap for each season (source: https://www.basketball-reference.com/contracts/salary-cap-history.html)
SALARY_CAP = {
    2017: 94143000,     2018: 99030000,     2019: 101869000,    2020: 109140000,    2021: 109140000,
    2022: 112414000,    2023: 123655000,    2024: 136021000,    2025: 140588000,    2026: 154647000
}

# Salary cap threshold based on salary cap percentage to filter out players with very low salaries (e.g. two-way contract players, 10-day contract players, etc.)
SALARY_CAP_THRESHOLD_PCT = 1  # Filter out players earning less than 1% of the salary cap

## Section 2 — Scrape a Single Team/Season

Defines `scrape_salaries_team_season(team, season)`, which fetches and parses the salary table for one team-season from Basketball Reference. Returns a DataFrame with `year`, `team`, `player`, and raw `salary` columns.

Basketball Reference renders salary tables inside HTML comment blocks — `BeautifulSoup`'s `Comment` filter is used to extract and re-parse them before locating the `#salaries2` table. A randomized 1–3 second delay is applied after each request. Run this against a single team/season to validate the scraper before kicking off the full pipeline.

In [29]:
def scrape_salaries_team_season(team: str, season: int) -> pd.DataFrame:
    """
    Scrapes the salaries of players for a given NBA team and season from Basketball Reference.
    Parameters:
        team (str): The three-letter abbreviation of the NBA team (e.g., 'LAL' for Los Angeles Lakers).
        season (int): The year of the season to scrape (e.g., 2022 for the 2021-2022 season).
    Returns:
        pd.DataFrame: A DataFrame containing the year, team name, player names, and their salaries for the specified team and season.
    """

    # Construct the URL for the specified team and season
    url = f'{BASE_URL}/{team}/{season}.html'

    # Make the HTTP request to the URL and parse the HTML content
    response = requests.get(url, headers = HEADERS)
    response.encoding = 'utf-8'  # Ensure the correct encoding for players with special characters in their names
    soup = BeautifulSoup(response.text, 'html.parser')

    # Basketball Reference often hides tables within HTML comments, so we need to extract those comments and parse them
    comments = soup.find_all(string = lambda text: isinstance(text, Comment))

    # Initialize an empty variable to hold the salaries table
    table = None

    # Loop through the comments to find the one that contains the salaries table
    for comment in comments:
        if 'div_salaries' in comment:
            comment_soup = BeautifulSoup(comment, 'html.parser')
            table = comment_soup.find('table', {'id': 'salaries2'})
            break

    # Error handling if salary table isn't found when request is made
    if table is None:
        print(f'No salary table found for {team} {season}')
        return pd.DataFrame(columns = ['year', 'team', 'player', 'salary'])

    # Loop through the rows of the table and extract the data into the DataFrame
    # tr: table row, td: table data, th: table header
    rows = []
    for row in table.find_all('tr')[1:]:
        cells = [cell.text.strip() for cell in row.find_all(['td', 'th'])]
        rows.append({'player': cells[1], 'salary': cells[2]})
    df = pd.DataFrame(rows)

    # Add the year and team name to the DataFrame
    df.insert(0, 'year', season)
    df.insert(1, 'team', team)

    # Add a delay to avoid overwhelming the server with requests 
    time.sleep(random.uniform(1, 3))  

    return df

In [31]:
# sanity check: test to validate that the single team/season scraper returns raw salary data correctly
scrape_salaries_team_season('LAL', 2024)

No salary table found for LAL 2024


AttributeError: 'Index' object has no attribute '_format_flat'

Empty DataFrame
Columns: [year, team, player, salary]
Index: []

## Section 3 — Scrape All Seasons

Defines `scrape_salaries_all_teams_seasons(start, end)`, which iterates over every team × season combination and concatenates results into a single raw DataFrame. Errors are caught per-team and surfaced at the end of each season's log line without interrupting the run.

> ⚠️ With 30 teams × 10 seasons and a 1–3s delay per request, expect a runtime of roughly **10 minutes**. Do not interrupt mid-run.

In [7]:
def scrape_salaries_all_teams_seasons(start_season: int = START_YEAR, end_season: int = END_YEAR) -> pd.DataFrame:
    """
    Scrapes the salaries of players for all NBA teams and seasons within a specified range from Basketball Reference.
    Parameters:
        start_season (int): The starting year of the season range to scrape (e.g., 2018 for the 2017-2018 season). Defaults to START_YEAR (2017).
        end_season (int): The ending year of the season range to scrape (e.g., 2022 for the 2021-2022 season). Defaults to END_YEAR (2026).
    Returns:
        pd.DataFrame: A DataFrame containing the year, team name, player names, and their salaries for all teams and seasons within the specified range.
    """
    # Initialize a list to hold DataFrames for each team/season
    all_dataframes = []

    # Loop through each season and team to scrape the salaries
    for season in range(start_season, end_season + 1):
        print(f'Scraping season {season - 1}-{str(season)[-2:]} ({season - start_season + 1}/{end_season - start_season + 1} seasons)...', end = ' ', flush = True)
        errors = []
        for team in TEAMS:
            try:
                team_data = scrape_salaries_team_season(team, season)
                all_dataframes.append(team_data)
            except Exception as e:
                errors.append((team))

        # Print the status of the scraping for the current season, including any errors encountered
        if len(errors) > 0:
            print(f'Done. Errors for teams: {", ".join(errors)}')
        else:
            print('Done.')

    # Concatenate all the individual DataFrames into a single DataFrame
    all_data = pd.concat(all_dataframes, ignore_index = True)
    
    # Final summary of the scraping process
    print(f'\nScraping complete. {len(all_data)} records collected across {end_season - start_season + 1} seasons.')
    return all_data

In [8]:
# run scraper to get all salaries within the set season range (defaulted to last 10 seasons
all_years = scrape_salaries_all_teams_seasons()

Scraping season 2016-17 (1/10 seasons)... Done.
Scraping season 2017-18 (2/10 seasons)... Done.
Scraping season 2018-19 (3/10 seasons)... Done.
Scraping season 2019-20 (4/10 seasons)... Done.
Scraping season 2020-21 (5/10 seasons)... Done.
Scraping season 2021-22 (6/10 seasons)... Done.
Scraping season 2022-23 (7/10 seasons)... Done.
Scraping season 2023-24 (8/10 seasons)... Done.
Scraping season 2024-25 (9/10 seasons)... Done.
Scraping season 2025-26 (10/10 seasons)... Done.

Scraping complete. 6535 records collected across 10 seasons.


In [9]:
# sanity check: preview complete scrape with a random selection of 5 players from any given season
print(f'Shape of scraped data: {all_years.shape}')
display(all_years.sample(5))

Shape of scraped data: (6535, 4)


,year,team,player,salary
757,2018,DAL,Yogi Ferrell,"$1,312,611"
3360,2022,HOU,D.J. Augustin,"$7,000,000"
256,2017,LAC,Wesley Johnson,"$5,628,000"
2760,2021,LAC,Amir Coffey,$0
4063,2023,GSW,Patrick Baldwin,"$2,226,000"


## Section 4 — Clean & Preprocess

Defines `clean_preprocess_data(df)`, which transforms the raw scraped data into a structured DataFrame.

### Steps:

1. **Salary parsing** — strips `$` and `,` from raw salary strings, coerces to numeric, and drops rows where parsing failed

2. **Season label** — derives a human-readable `season` string from `year` (e.g. `2022` → `2021-22`)

3. **Multi-team aggregation** — collapses players who appeared on multiple teams into a single `TOT` row with salary summed across stints

4. **Cap normalization** — maps each season's salary cap from `SALARY_CAP` and calculates `salary_cap_pct`

5. **Unique ID** — constructs a `{first_initial}_{last_name}_{team}_{year}` index (e.g. `l_james_lal_24`)

6. **Low-salary filter** — removes players earning below `SALARY_CAP_THRESHOLD_PCT` to exclude minimum and two-way contract players

In [32]:
SUFFIXES = {'jr', 'jr.', 'sr', 'sr.', 'ii', 'iii', 'iv', 'v'}

def build_id(name: str, team: str, year: str) -> str:
    # Builds a unique player-season identifier in the format 'leb_james_lal_24'
    # Normalize unicode characters to ASCII (e.g. Jokić -> Jokic)
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    tokens = name.lower().split()
    tokens = [t for t in tokens if t not in SUFFIXES]
    if not tokens:
        return f'unknown_{team.lower()}_{str(year)[-2:]}'
    first = tokens[0][:4]
    last = tokens[-1]
    return f'{first}_{last}_{team.lower()}_{str(year)[-2:]}'

def clean_preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans and preprocesses the scraped salary data.
    Parameters:
        df (pd.DataFrame): The raw DataFrame containing the scraped salary data.
    Returns:
        pd.DataFrame: A cleaned and preprocessed DataFrame ready for analysis.
    Cleaned DataFrame will include:
        - 'id': A unique identifier for each player-season, created by combining the player's name, team, and season.
        - 'player': The name of the player.
        - 'team': The three-letter abbreviation of the team (e.g., 'LAL' for Los Angeles Lakers).
        - 'year': The year of the season (e.g., 2022 for the 2021-2022 season).
        - 'season': A string representing the season in the format 'YYYY-YY' (e.g., '2021-22' for the 2021-2022 season).
        - 'salary': The player's salary for that season, converted to an integer.
        - 'salary_cap': The salary cap for that season, mapped from a predefined dictionary of salary caps by year.
        - 'salary_cap_pct': The percentage of the salary cap that the player's salary represents.
    """

    # Create a copy of the DataFrame to avoid modifying the original data
    df = df.copy()

    # Clean the 'Salary' column by removing dollar signs and commas, and convert it to numeric type
    df['salary'] = pd.to_numeric(df['salary'].str.replace(r'[$,]', '', regex = True), errors = 'coerce')

    # Drop rows where 'Salary' is missing (NaN) after conversion - Basketball Reference occasionally has missing salary data for low-profile players
    df = df.dropna(subset = ['salary'])

    # Convert the 'Year' column to numeric
    df['year'] = df['year'].astype(int)

    # Create a 'Season' column in the format 'YYYY-YY' (e.g., '2021-22' for the 2021-2022 season)
    df['season'] = df['year'].apply(lambda x: f'{x-1}-{str(x)[-2:]}')

    # Aggregate salaries for players who played for multiple teams in the same season by summing their salaries
    df = df.groupby(['season', 'year', 'player'], as_index = False).agg({
        'team': lambda x: 'TOT' if len(x) > 1 else x.iloc[0],  # Indicate if the player switched teams during the season (Basketball Reference uses 'TOT' for total when a player switches teams)
        'salary': 'sum',  # Sum salaries for players who switched teams
    })

    # Map the salary caps to the DataFrame based on the 'Year' column
    df['salary_cap'] = df['year'].map(SALARY_CAP).astype('float')

    # Calculate the percentage of the salary cap that each player's salary represents
    df['salary_cap_pct'] = round((df['salary'] / df['salary_cap']) * 100, 2)

    # Drop rows with missing or empty player names
    df = df[df['player'].str.strip() != '']
    df = df.dropna(subset = ['player'])

    # Create a unique identifier for each player-season 
    df['id'] = df.apply(lambda x: build_id(x['player'], x['team'], x['year']), axis = 1)

    # Remove players with very low salaries (less than the defined salary cap threshold percentage)
    df = df[df['salary_cap_pct'] >= SALARY_CAP_THRESHOLD_PCT]

    # Replace dataframe index with custom id feature
    df.set_index('id', inplace = True)

    # Reorder the columns for better readability
    df = df[['player', 'team', 'year', 'season', 'salary', 'salary_cap', 'salary_cap_pct']]

    return df

In [33]:
# run clean/preprocessing function to clean player salary data and get a random selection of 5 players from any given season to preview dataset
cleaned_data = clean_preprocess_data(all_years)
display(cleaned_data.sample(5))

,player,team,year,season,salary,salary_cap,salary_cap_pct
id,,,,,,,
tren_watford_phi_26,Trendon Watford,PHI,2026,2025-26,2461463.0,154647000.0,1.59
toma_satoransky_was_19,Tomáš Satoranský,WAS,2019,2018-19,3129187.0,101869000.0,3.07
will_cauley-stein_dal_20,Willie Cauley-Stein,DAL,2020,2019-20,2177483.0,109140000.0,2.00
tyle_johnson_mia_17,Tyler Johnson,MIA,2017,2016-17,5628000.0,94143000.0,5.98
mich_porter_den_20,Michael Porter,DEN,2020,2019-20,3389400.0,109140000.0,3.11


## Section 5 — Feature Engineering

Defines `feature_engineering(df)`, which builds contract-status and salary context features on top of the cleaned data.

### Features added:

1. **`salary_tier`** — categorical label (bench / starter / star / max) based on cap % buckets: `0–8%`, `8–15%`, `15–25%`, `25%+`

2. **`salary_pct_change`** — year-over-year salary change percentage, calculated per player using `pct_change()`

3. **`is_max`** — flags players earning ≥ 25% of the salary cap

4. **`is_new_team`** — flags players whose team differs from their prior season record, indicating a free agent signing or trade

5. **`is_contract_year`** — flags players who signed a new deal the following season, inferred from a >20% salary jump in year+1

6. **`is_moved_midseason`** — flags players with `team == TOT`; salary rank is nulled out for these players since it's ambiguous across stints

In [34]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Performs feature engineering on the cleaned salary data to create additional features for analysis.
    Parameters:
        df (pd.DataFrame): The cleaned DataFrame containing the salary data.
    Returns:
        pd.DataFrame: A DataFrame with additional engineered features for analysis.
    Engineered features will include:
        - 'salary_tier': A categorical variable that classifies players into salary cap percentage categories (e.g., '0-10%', '10-25%', '25%+) based on their 'Salary Cap Pct' value.
        - 'salary_pct_change': A feature that calculates the percentage change in salary from the previous season to the current season, providing insight into the player's salary trajectory.
        - 'is_max': A binary variable indicating whether the player's salary is at or above a certain threshold (e.g., 25% of the salary cap) to identify max contract players.
        - 'is_new_team': A binary variable indicating whether the player is on a new team based on their team from the previous season's record.
        - 'is_contract_year': A binary variable indicating whether the player is in a contract year, which can be inferred from the player's contract history and the current season.
        - 'is_moved_midseason': A binary variable indicated whether the player was traded/moved mid-season between teams
    """

    # Create a copy of the DataFrame to avoid modifying the original data
    df = df.copy()

    # Sort the DataFrame by player and season to facilitate the creation of previous season features
    df.sort_values(by = ['player', 'year'], inplace = True)

    # Variable to classify players into salary cap percentage categories
    df['salary_tier'] = pd.cut(
        df['salary_cap_pct'], 
        bins = [0, 8, 15, 25, 100], 
        labels = ['bench', 'starter', 'star', 'max']
    )

    # Flag to identify max contract players (e.g., those earning 25% or more of the salary cap)
    df['is_max'] = (df['salary_cap_pct'] >= 25).astype(int)

    # Year-over-year salary change percentage
    df['salary_pct_change'] = df.groupby('player')['salary'].pct_change() * 100

    # Flag to identify if the player is on a new team compared to the previous season (indicating a potential new contract)
    df['is_new_team'] = ((df.groupby('player')['team'].shift(1) != df['team']) & (df['salary_pct_change'].notna())).astype(int)

    # Flag to identify if the player is in a contract year which is inferred from a player's salary in the following season
    df['is_contract_year'] = (df.groupby('player')['salary_pct_change'].shift(-1) > 20).astype(int)

    # Flag to identify if the player was traded/moved mid-season
    df['is_moved_midseason'] = (df['team'] == 'TOT').astype(int)

    df = df[['player', 'team', 'year', 'season', 'salary', 'salary_cap', 'salary_cap_pct', 'salary_tier', 'salary_pct_change', 
             'is_max', 'is_new_team', 'is_contract_year', 'is_moved_midseason']]

    return df

In [35]:
# run feature engineering function to derive additional features and get a random selection of 5 players from any given season to preview dataset
enhanced_data = feature_engineering(cleaned_data)
display(enhanced_data.sample(5))

,player,team,year,season,salary,salary_cap,salary_cap_pct,salary_tier,salary_pct_change,is_max,is_new_team,is_contract_year,is_moved_midseason
id,,,,,,,,,,,,,
obi_toppin_nyk_22,Obi Toppin,NYK,2022,2021-22,5105160.0,112414000.0,4.54,bench,5.000370,0,0,0,0
drew_eubanks_lac_25,Drew Eubanks,LAC,2025,2024-25,5000000.0,140588000.0,3.56,bench,113.072964,0,1,0,0
yogi_ferrell_dal_18,Yogi Ferrell,DAL,2018,2017-18,1312611.0,99030000.0,1.33,bench,NaN,0,0,1,0
jord_nwora_ind_23,Jordan Nwora,IND,2023,2022-23,2800000.0,123655000.0,2.26,bench,84.455537,0,1,0,0
cody_zeller_cho_21,Cody Zeller,CHO,2021,2020-21,15415730.0,109140000.0,14.12,starter,6.521738,0,0,0,0


## Section 6 — Export to CSV

Exports the final enhanced DataFrame to `nba_salaries.csv` at the path defined by `OUTPUT_FILE`. Prints a confirmation with the total record count. The exported file includes the `id` index and all columns from the output schema above.

In [36]:
def export_to_csv(df: pd.DataFrame, filepath: str = OUTPUT_FILE) -> None:
    """
    Exports the DataFrame to a CSV file.
    Parameters:
        - df (pd.DataFrame): The DataFrame to export
        - filepath (str): The path to the output CSV file. Defaults to OUTPUT_FILE
    Returns:
        - None
    """
    df.to_csv(filepath, index = True)
    print(f'Data exported to \'{filepath}\' - {df.shape}')

In [37]:
# run csv export function to save enhanced salary dataset to .csv file for analysis
export_to_csv(enhanced_data, './data/nba_player_salaries.csv')

Data exported to './data/nba_player_salaries.csv' - (4420, 13)


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fa7bfbce-9eb0-4b44-99aa-9c5ae30d8022' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>